# EDA GA4

**Caso:** Google Merchandise Store, usando a amostra pública de eventos GA4.

Nesta aula vamos investigar como visitantes navegam, buscam produtos e compram. O objetivo não é aprender somente a sintaxe mas como transformar evidências em decisões.

## 1. CRISP-DM

O CRISP-DM organiza um projeto analítico em seis etapas: **Business Understanding**, **Data Understanding**, **Data Preparation**, **Modeling**, **Evaluation** e **Deployment**.

Hoje vamos concluir o alinhamento do problema e entrar na exploração dos dados. Em outras palavras: transformar uma ambição como *aumentar conversão* em hipóteses mensuráveis e verificar se a base permite investigá-las.

## 2. Fase 1: Business Understanding

**Problema de negócio:** compreender o comportamento dos visitantes para aumentar a taxa de conversão.

### Hipóteses para investigar
1. **Dispositivo:** a conversão pode ser diferente entre mobile, desktop e tablet; uma diferença grande pode sinalizar oportunidade de UX mobile.
2. **Busca e intenção:** visitantes que usam a busca interna podem converter mais; esse grupo pode orientar mídia e melhorias na descoberta de produtos.
3. **Profundidade:** visitantes que percorrem mais páginas ou eventos podem demonstrar maior interesse, mas navegação excessiva também pode indicar fricção.

Uma hipótese útil sempre explicita: **segmento**, **métrica**, **comparação** e **decisão possível**.

> 💡 **Pergunta para Discussão:** Para cada hipótese, qual seria a decisão executiva se a evidência fosse forte? E qual seria o risco de agir com base em uma correlação fraca?

### Exercício 1:

Preencha a tabela abaixo com uma métrica e uma decisão para cada hipótese.

In [ ]:
hipoteses = {
    "Dispositivo": {"metrica": "", "decisao": ""},
    "Busca e intenção": {"metrica": "", "decisao": ""},
    "Profundidade": {"metrica": "", "decisao": ""},
}
hipoteses

## 3. Preparação do ambiente no Google Colab

O BigQuery fornece a origem dos dados. Depois da extração, trabalharemos com um DataFrame cacheado localmente para repetir análises sem consultar novamente a nuvem. O projeto informado é usado para processamento e pode estar sujeito às regras de cobrança, caso não use o ambiente sandbox do GCP.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

In [ ]:
# Caso esteja usando o VSCode com o Colab, rode o seguinte para habilitar o plotly no VSCode:
#import plotly.io as pio
#pio.renderers.default = "vscode"

In [ ]:
project_id = input("Informe o ID do seu projeto Google Cloud: " ).strip()
if not project_id:
    raise ValueError("Informe um project_id para executar consultas no BigQuery.")

client = bigquery.Client(project=project_id)
print(f"Cliente BigQuery pronto para o projeto: {project_id}")

In [ ]:
# Janela curta para a aula: altere as datas para expandir a análise.
START_DATE = "2021-01-01"
END_DATE = "2021-01-14"
CACHE_PATH = Path(f"ga4_events_cache_{START_DATE}_{END_DATE}.parquet")

if START_DATE > END_DATE:
    raise ValueError("START_DATE deve ser anterior ou igual a END_DATE.")

print(f"Período selecionado: {START_DATE} a {END_DATE}")

A tabela pública é `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`. O `*` representa uma tabela por dia; por isso filtramos `_TABLE_SUFFIX` para controlar custo. O GA4 armazena parâmetros em estruturas repetidas, como `event_params`, que são abertas com `UNNEST`.

Wildcard tables tem várias limitações, para novos usos recomenda-se o uso de partitioning e clustering.

Antes de executar uma consulta, fazemos um **dry run**: ele estima os bytes processados sem retornar linhas. Isso é uma prática de governança, não apenas uma preocupação técnica.

In [ ]:
QUERY = """
SELECT
  event_date,
  event_name,
  user_pseudo_id,
  event_timestamp,
  device.category AS device_category,
  (
    SELECT value.int_value
    FROM UNNEST(event_params)
    WHERE key = 'ga_session_id'
    LIMIT 1
  ) AS ga_session_id,
  (
    SELECT value.string_value
    FROM UNNEST(event_params)
    WHERE key = 'search_term'
    LIMIT 1
  ) AS search_term
FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE _TABLE_SUFFIX BETWEEN REPLACE(@start_date, '-', '') AND REPLACE(@end_date, '-', '')
"""

job_config = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ScalarQueryParameter("start_date", "STRING", START_DATE),
    bigquery.ScalarQueryParameter("end_date", "STRING", END_DATE),
])
dry_config = bigquery.QueryJobConfig(
    query_parameters=job_config.query_parameters,
    dry_run=True,
    use_query_cache=False,
)
dry_job = client.query(QUERY, job_config=dry_config)
bytes_processed = dry_job.total_bytes_processed or 0
print(f"Estimativa do dry run: {bytes_processed / 1024**3:.2f} GB processados")

### Exercício 2:

> 💡 **Pergunta para Discussão:** Que decisão de negócio pode ser comprometida quando a organização não controla o custo, o período e a origem da consulta?

> 📝 **Seu Insight**:

In [ ]:

if CACHE_PATH.exists():
    events = pd.read_parquet(CACHE_PATH)
    print(f"Cache reutilizado: {CACHE_PATH} ({len(events):,} eventos)")
else:
    events = client.query(QUERY, job_config=job_config).to_dataframe()
    events.to_parquet(CACHE_PATH, index=False)
    print(f"Consulta concluída e cache salvo: {CACHE_PATH} ({len(events):,} eventos)")

In [ ]:
events["event_date"] = pd.to_datetime(events["event_date"], format="%Y%m%d", errors="coerce")
events["user_pseudo_id"] = events["user_pseudo_id"].fillna("unknown_user")
events["device_category"] = events["device_category"].fillna("unknown")
events["ga_session_id"] = events["ga_session_id"].astype("Int64")
events["session_key"] = (
    events["user_pseudo_id"] + "-" + events["ga_session_id"].astype("string")
)
events["is_purchase"] = events["event_name"].eq("purchase")
events["has_search"] = events["event_name"].eq("view_search_results") | events["search_term"].notna()
events["has_add_to_cart"] = events["event_name"].eq("add_to_cart")
events["is_page_view"] = events["event_name"].eq("page_view")

display(events.head())
print(f"Eventos: {len(events):,} | Usuários: {events['user_pseudo_id'].nunique():,}")

## 4. Visão macro e baseline

Vamos primeiro construir uma referência. Sem baseline, uma diferença entre segmentos pode parecer importante apenas porque não sabemos qual é o nível geral da loja.

A conversão será apresentada por usuário e por sessão. São lentes diferentes: a primeira responde *quantas pessoas compraram?*; a segunda responde *quantas visitas chegaram à compra?*

In [ ]:
user_summary = events.groupby("user_pseudo_id", as_index=False).agg(
    sessions=("session_key", "nunique"),
    event_count=("event_name", "size"),
    page_views=("is_page_view", "sum"),
    searched=("has_search", "max"),
    added_to_cart=("has_add_to_cart", "max"),
    purchased=("is_purchase", "max"),
)

session_summary = events.groupby("session_key", dropna=False, as_index=False).agg(
    user_pseudo_id=("user_pseudo_id", "first"),
    device_category=("device_category", "first"),
    event_count=("event_name", "size"),
    page_views=("is_page_view", "sum"),
    searched=("has_search", "max"),
    added_to_cart=("has_add_to_cart", "max"),
    purchased=("is_purchase", "max"),
)

print("\nMétricas de baseline:")
print(f"Usuário: {len(user_summary):,}")
print(f"Sessões: {len(session_summary):,}")
print(f"Eventos: {len(events):,}")
print(f"Compradores: {int(user_summary['purchased'].sum()):,}")
print(f"Sessões convertidas: {int(session_summary['purchased'].sum()):,}")
print(f"Conversão por usuário: {user_summary['purchased'].mean():.2%}")
print(f"Conversão por sessão: {session_summary['purchased'].mean():.2%}")


### Exercício 3:

> 💡 **Pergunta para Discussão:** Se 99% dos usuários não compram, um modelo que sempre prevê “não compra” teria alta acurácia. Por que isso seria uma métrica perigosa para decidir onde investir?

A acurácia mistura a classe majoritária com a minoritária. Em conversão, precisamos observar a taxa de compra, o volume potencial, o custo da intervenção e métricas por segmento. Um resultado raro pode ter alto valor econômico.


> 📝 **Seu Insight:**

### Exercício 4
Calcule a participação de usuários que fizeram uma busca interna e a participação de usuários que adicionaram um produto ao carrinho.

In [ ]:
# Complete as duas expressões abaixo.
percentual_busca = None
percentual_carrinho = None
print(percentual_busca, percentual_carrinho)

## 5. Hipótese 1: dispositivo versus conversão

> 💡 **Pergunta para Discussão:** A conversão mobile está abaixo da desktop?

A tabela abaixo mostra volume e taxa. Volume evita priorizar um segmento pequeno apenas porque sua taxa parece extrema.

In [ ]:
device_analysis = session_summary.groupby("device_category", as_index=False).agg(
    sessions=("session_key", "size"),
    converted_sessions=("purchased", "sum"),
    users=("user_pseudo_id", "nunique"),
)
device_analysis["session_conversion"] = device_analysis["converted_sessions"] / device_analysis["sessions"]
device_analysis = device_analysis.sort_values("session_conversion", ascending=False)
display(device_analysis.style.format({"session_conversion": "{:.3%}"}))

In [ ]:
fig = px.bar(
    device_analysis,
    x="device_category",
    y="session_conversion",
    color="device_category",
    text="session_conversion",
    hover_data=["sessions", "converted_sessions", "users"],
    title="Conversão por sessão e dispositivo",
    labels={"device_category": "Dispositivo", "session_conversion": "Conversão por sessão"},
)
fig.update_traces(texttemplate="%{text:.4%}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_tickformat=".1%")
fig.show()

### Exercício 5
Crie um gráfico equivalente comparando o número absoluto de sessões por dispositivo.

In [ ]:
fig_volume = px.bar(None)
fig_volume.show()

## 6. Hipótese 2: eventos de alta intenção

> 💡 **Pergunta para Discussão:** Usuários que buscam convertem mais?

Vamos criar grupos mutuamente exclusivos no nível de usuário.

In [ ]:
def definir_grupo_intencao(linha):
    if linha["searched"]:
        return "Fez busca"
    if linha["added_to_cart"]:
        return "Adicionou ao carrinho"
    return "Apenas navegou (erro?!)"

user_summary["intent_group"] = user_summary.apply(definir_grupo_intencao, axis=1)
intent_analysis = user_summary.groupby("intent_group", as_index=False).agg(
    users=("user_pseudo_id", "size"),
    buyers=("purchased", "sum"),
)
intent_analysis["user_conversion"] = intent_analysis["buyers"] / intent_analysis["users"]
intent_analysis = intent_analysis.sort_values("user_conversion", ascending=False)
display(intent_analysis.style.format({"user_conversion": "{:.2%}"}))

In [ ]:
fig = px.bar(
    intent_analysis,
    x="intent_group",
    y="user_conversion",
    color="intent_group",
    text="user_conversion",
    hover_data=["users", "buyers"],
    title="Conversão por grupo de intenção",
    labels={"intent_group": "Comportamento", "user_conversion": "Conversão por usuário"},
)
fig.update_traces(texttemplate="%{text:.2%}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_tickformat=".0%")
fig.show()

Um grupo com alta conversão pode ser um excelente alvo de experiência e remarketing. Mas a análise é observacional: não sabemos se a busca causou a compra ou se compradores mais motivados simplesmente buscaram mais.

### Exercício 6
Altere o código para considerar também quando há `searched` e `added_to_cart`, em vez de usar somente grupos exclusivos. Qual abordagem é mais útil para uma decisão de produto?

In [ ]:
# Dica: use o código acima ou então faça groupby nas duas colunas booleanas e calcule a média de purchased. 
intencao_longa = user_summary.groupby(None, as_index=False).agg(
    users=("user_pseudo_id", "size"),
    conversion=None,
)

intencao_longa = intencao_longa.sort_values("conversion", ascending=False)
display(intencao_longa.style.format({"conversion": "{:.2%}"}))

## 7. Hipótese 3: profundidade de navegação

> 💡 **Pergunta para Discussão:** Mais páginas vistas significam maior interesse ou mais dificuldade para encontrar o produto certo? Em que faixa a experiência parece mudar?

Usaremos faixas simples de páginas por usuário. Faixas tornam o resultado mais fácil de discutir, mas perdem algum detalhe.

In [ ]:
bins = [-1, 1, 3, 7, float("inf")]
labels = ["0-1 página", "2-3 páginas", "4-7 páginas", "7+ páginas" ]
user_summary["depth_group"] = pd.cut(user_summary["page_views"], bins=bins, labels=labels)
depth_analysis = user_summary.groupby("depth_group", observed=False, as_index=False).agg(
    users=("user_pseudo_id", "size"),
    buyers=("purchased", "sum"),
)
depth_analysis["user_conversion"] = depth_analysis["buyers"] / depth_analysis["users"]
display(depth_analysis.style.format({"user_conversion": "{:.2%}"}))

In [ ]:
fig = px.bar(
    depth_analysis,
    x="depth_group",
    y="user_conversion",
    color="depth_group",
    text="user_conversion",
    hover_data=["users", "buyers"],
    title="Conversão por profundidade de navegação",
    labels={"depth_group": "Páginas vistas", "user_conversion": "Conversão por usuário"},
)
fig.update_traces(texttemplate="%{text:.2%}", textposition="outside")
fig.update_layout(showlegend=False, yaxis_tickformat=".0%")
fig.show()

### Exercício 7
Modifique o código acima e inclua novos bins e labels para a variável depth_group, de forma a ter mais granularidade na análise. 
Por exemplo, você pode criar grupos como "0-1 página", "2-3 páginas", "4-7 páginas", "8-10 páginas", "11-15 páginas", "16-20 páginas", "21-40 páginas", "41-80 páginas" e "81+ páginas".

In [ ]:
bins = None
labels = None
user_summary["depth_group"] = pd.cut(user_summary["page_views"], bins=bins, labels=labels)
depth_analysis = user_summary.groupby("depth_group", observed=False, as_index=False).agg(
    users=("user_pseudo_id", "size"),
    buyers=("purchased", "sum"),
)
depth_analysis["user_conversion"] = depth_analysis["buyers"] / depth_analysis["users"]
display(depth_analysis.style.format({"user_conversion": "{:.2%}"}))

## 8. Mapa de correlação

> 💡 **Pergunta para Discussão:** Quais comportamentos têm relação mais forte com a compra? Uma correlação alta, sozinha, justifica investimento?

Vamos correlacionar comportamentos agregados no nível de usuário. A flag `purchased` é o resultado que queremos entender; as outras colunas são sinais possíveis. Correlação ajuda a priorizar investigação, mas não prova causalidade nem ROI.

Nota: A correlação é uma medida estatística que indica a força e a direção da relação entre duas variáveis. Ela mostra o quanto essas variáveis se movem juntas

In [ ]:
correlation_data = user_summary[[
    "event_count", "sessions", "page_views",
    "searched", "added_to_cart", "purchased"
]].astype(float)
correlation_matrix = correlation_data.corr()

fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Correlação entre comportamentos de usuário",
)
fig.update_layout(coloraxis_colorbar_title="Correlação")
fig.show()

**Como separar ação de vaidade:** um sinal é mais acionável quando tem volume relevante, ligação plausível com a jornada, possibilidade de intervenção e uma forma de medir impacto. O número de eventos, isoladamente, pode crescer sem aumentar receita.

### Exercício 8
Escolha uma métrica do mapa de correlação e escreva uma recomendação que inclua: público afetado, ação, métrica de sucesso e risco da decisão.

In [ ]:
recomendacao = {
    "publico": "",
    "acao": "",
    "metrica_de_sucesso": "",
    "risco": "",
}
recomendacao

## 9. Fechamento

Uma boa conclusão não é “o gráfico subiu”. É uma decisão testável com impacto potencial, esforço estimado e nível de confiança.

### Sugestão de Checklist
- Qual segmento apresenta a maior oportunidade absoluta?
- Qual comportamento parece mais próximo da intenção de compra?
- Que resultado pode ser explicado por viés de seleção ou volume pequeno?
- Qual teste ou análise adicional reduziria mais a incerteza?

### Limitações importantes
- Observamos associação, não causalidade.
- O período curto pode produzir taxas instáveis em segmentos pequenos.
